## 06. COPY INTO And MERGE Commands — objective

- Trace key-based changes from incoming records into a persistent target.
- Separate file discovery, parsing and schema tracking.

Historical results; no code executed.
- [Original inputs: stock_prices](../../../data/stock_prices/) — 3 files.

# 06. COPY INTO And MERGE Commands

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea04-delta-lake/06.%20COPY%20INTO%20And%20MERGE%20Commands.json)

## Learning notes — before source operation

**merge**: MERGE matches source rows to target keys and applies matched/unmatched actions. Verify source-key uniqueness and distinguish updates from inserts.

# COPY INTO and MERGE Commands

### COPY INTO Command

> - Incrementally loads data into Delta Lake tables from Cloud Storage  
> - Supports schema evolution  
> - Supports wide range of file formats (CSV, JSON, Parquet, Delta)  
> - Alternative to Auto Loader for batch ingestion  

Documentation - [COPY INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)

#### Create the table to copy the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

#### Incrementally load new files into the table

Archived Databricks source — shown for reading, not Python execution.

```sql
DELETE FROM demo.delta_lake.raw_stock_prices;

COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@deacourseextdl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

In [ ]:
# Archived output only; no computation.

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
6,6,0


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

price,status,stock_id,trading_date
233.5,ACTIVE,AAPL,2025-02-12
2805.0,ACTIVE,GOOGL,2025-02-12
335.0,ACTIVE,MSFT,2025-02-12
3520.0,ACTIVE,AMZN,2025-02-12
405.0,ACTIVE,META,2025-02-12
null,DELISTED,TSLA,2025-02-12


### MERGE Statement
> - Used for upserts (Insert/ Update/ Delete operations in a single statement)
> - Allows merging new data into a target table based on matching condition

Documentation - [MERGE INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-merge-into)

#### Create the table to merge the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
  stock_id STRING,
  price DOUBLE,
  trading_date DATE
);
```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

#### Merge the source data into target table
1. Insert new stocks received
2. Update price and trading_date if updates receieved
3. Delete stocks which are de-listed from the exchange (status = 'DELISTED')

Archived Databricks source — shown for reading, not Python execution.

```sql
MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source
   ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN
    UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN    
    DELETE
WHEN NOT MATCHED AND source.status ='ACTIVE' THEN     
    INSERT (stock_id, price, trading_date) VALUES (source.stock_id, source.price, source.trading_date);
```

In [ ]:
# Archived output only; no computation.

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,4,1,1


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.stock_prices;
```

In [ ]:
# Archived output only; no computation.

stock_id,price,trading_date
AAPL,233.5,2025-02-12
GOOGL,2805.0,2025-02-12
MSFT,335.0,2025-02-12
AMZN,3520.0,2025-02-12
META,405.0,2025-02-12
